# VectorNet H3 coverage maps

Interactive Folium maps of **H3 resolution-4** cells occupied by VectorNet detections in `data/02_interim/vectors_normalized.parquet`.

**What hexes mean:** a cell is drawn if VectorNet has ≥1 dated detection there (after optional filters). Empty space is *not* proven vector absence — only lack of records in this extract.

**Metrics are stored per `(h3_cell, vector_group)`:**
- `n_year_months` — how often detections appear (distinct year×month)
- `n_years` / `year_span` — how long the record spans
- `n_records` — raw detection row count (effort-sensitive)

**Map UX:** one polygon per H3 cell on an OpenStreetMap street basemap (hex fill is semi-transparent). Fill color uses the **max** of the plotted metric across groups in that cell (shared color scale).

**Hover or click** a hex for a **horizontal** table (metric rows × one column per vector group). When several groups are present, the **layer control** (top-right) toggles vector groups only (H3 polygons stay always on): unchecked groups disappear from the popup, and hexes that have **no remaining active group** are hidden.

Set `VECTOR_GROUP` to filter the dataset to one group, or `None` for all groups.

After pulling code changes: **restart the kernel** and re-run from the top so `vector_guard.viz` reloads.

Edit the **config** cell (`COUNTRY_ISO2`, `VECTOR_GROUP`) and re-run from there to redraw.

In [1]:
from pathlib import Path

import pandas as pd

from vector_guard.data import default_interim_dir, find_repo_root
from vector_guard.viz import aggregate_interim_vector_cells, folium_h3_choropleth

REPO_ROOT = find_repo_root(Path.cwd())
VECTOR_PATH = default_interim_dir(REPO_ROOT) / "vectors_normalized.parquet"
vectors = pd.read_parquet(VECTOR_PATH)
print(f"Loaded {len(vectors):,} rows from {VECTOR_PATH.relative_to(REPO_ROOT)}")
print("Countries:", vectors["country_iso2"].nunique())
print("Vector groups:", sorted(vectors["vector_group"].dropna().unique()))

Loaded 122,789 rows from data/02_interim/vectors_normalized.parquet
Countries: 57
Vector groups: ['biting_midge', 'mosquito', 'sandfly', 'tick']


## Config

Set `COUNTRY_ISO2` to an ISO-2 code (e.g. `"ES"`) to filter, or `None` for all countries.  
Set `VECTOR_GROUP` to `"mosquito"`, `"tick"`, `"biting_midge"`, or `"sandfly"` for a single group, or `None` to include all groups (multi-group cells list every group in the click popup).

In [2]:
COUNTRY_ISO2 = None  # e.g. "ES"; None = all countries
VECTOR_GROUP = None  # e.g. "tick"; None = all groups

In [3]:
filtered = vectors.copy()
if COUNTRY_ISO2 is not None:
    code = str(COUNTRY_ISO2).strip().upper()
    filtered = filtered.loc[filtered["country_iso2"].astype(str).str.upper() == code]
if VECTOR_GROUP is not None:
    group = str(VECTOR_GROUP).strip()
    filtered = filtered.loc[filtered["vector_group"].astype(str) == group]

cell_stats = aggregate_interim_vector_cells(
    vectors,
    country_iso2=COUNTRY_ISO2,
    vector_group=VECTOR_GROUP,
)

n_cells = int(cell_stats["h3_cell"].nunique()) if not cell_stats.empty else 0
n_multi = (
    int((cell_stats.groupby("h3_cell").size() >= 2).sum()) if not cell_stats.empty else 0
)

print(
    f"Filter: country={COUNTRY_ISO2!r}, vector_group={VECTOR_GROUP!r}"
)
print(f"Rows after filter: {len(filtered):,} / {len(vectors):,}")
print(f"(h3_cell, vector_group) units: {len(cell_stats):,}")
print(f"Unique H3 cells: {n_cells:,}")
print(f"H3 cells with ≥2 vector groups: {n_multi:,}")
if not cell_stats.empty:
    print("Units by vector_group:")
    display(cell_stats["vector_group"].value_counts())
    display(
        cell_stats[["n_year_months", "n_years", "year_span", "n_records"]].describe()
    )

Filter: country=None, vector_group=None
Rows after filter: 122,789 / 122,789
(h3_cell, vector_group) units: 2,690
Unique H3 cells: 2,200
H3 cells with ≥2 vector groups: 438
Units by vector_group:


vector_group
mosquito        1352
tick             594
biting_midge     530
sandfly          214
Name: count, dtype: int64

,n_year_months,n_years,year_span,n_records
count,2690.000000,2690.000000,2690.000000,2690.000000
mean,4.963941,2.123420,3.995167,45.277323
std,14.745965,2.429751,7.031379,333.205628
min,1.000000,1.000000,1.000000,1.000000
25%,1.000000,1.000000,1.000000,2.000000
50%,2.000000,1.000000,1.000000,6.000000
75%,4.000000,2.000000,3.000000,17.000000
max,461.000000,41.000000,66.000000,11389.000000


## Frequency map (`n_year_months`)

Fill color = **max** `n_year_months` across vector groups in the cell. Hover/click a hex for a horizontal stats table (one column per vector group).

In [4]:
freq_title = "n_year_months"
if COUNTRY_ISO2:
    freq_title += f" · {str(COUNTRY_ISO2).upper()}"
if VECTOR_GROUP:
    freq_title += f" · {VECTOR_GROUP}"

folium_h3_choropleth(cell_stats, "n_year_months", title=freq_title)

## Duration map (`n_years`)

Fill color = **max** `n_years` across vector groups in the cell. Hover/click a hex for a horizontal stats table (one column per vector group).

In [5]:
dur_title = "n_years"
if COUNTRY_ISO2:
    dur_title += f" · {str(COUNTRY_ISO2).upper()}"
if VECTOR_GROUP:
    dur_title += f" · {VECTOR_GROUP}"

folium_h3_choropleth(cell_stats, "n_years", title=dur_title)

## Top cells

Highest frequency and longest duration under the current filter (one row per `h3_cell` × `vector_group`).

In [6]:
cols = [
    "h3_cell",
    "vector_group",
    "n_year_months",
    "n_years",
    "year_span",
    "n_records",
    "year_min",
    "year_max",
]

print("Top by n_year_months")
display(cell_stats[cols].head(15))

print("Top by n_years")
display(
    cell_stats.sort_values(
        ["n_years", "year_span", "n_year_months"],
        ascending=False,
    )[cols].head(15)
)

Top by n_year_months


,h3_cell,vector_group,n_year_months,n_years,year_span,n_records,year_min,year_max
0,8419519ffffffff,biting_midge,461,41,41,11389,1974,2014
1,84195b7ffffffff,biting_midge,338,40,41,8673,1974,2014
2,84194e7ffffffff,mosquito,110,24,42,532,1980,2021
3,84195b7ffffffff,mosquito,101,16,28,409,1992,2019
4,84195b9ffffffff,mosquito,94,20,40,302,1982,2021
5,84194e3ffffffff,mosquito,82,27,40,278,1980,2019
6,841e845ffffffff,biting_midge,78,7,7,912,2001,2007
7,841eb13ffffffff,biting_midge,75,7,7,900,2001,2007
8,841136bffffffff,mosquito,73,13,13,702,2010,2022
9,841e805ffffffff,biting_midge,72,9,22,358,2001,2022


Top by n_years


,h3_cell,vector_group,n_year_months,n_years,year_span,n_records,year_min,year_max
0,8419519ffffffff,biting_midge,461,41,41,11389,1974,2014
1,84195b7ffffffff,biting_midge,338,40,41,8673,1974,2014
5,84194e3ffffffff,mosquito,82,27,40,278,1980,2019
2,84194e7ffffffff,mosquito,110,24,42,532,1980,2021
4,84195b9ffffffff,mosquito,94,20,40,302,1982,2021
55,84194adffffffff,mosquito,42,20,38,61,1980,2017
25,8419597ffffffff,mosquito,60,19,40,269,1980,2019
50,84195bbffffffff,mosquito,44,17,41,148,1980,2020
22,841948dffffffff,mosquito,64,17,39,195,1982,2020
3,84195b7ffffffff,mosquito,101,16,28,409,1992,2019
